# Разработка адаптера BLP -> MCLP

In [1]:
import osmnx as ox
import numpy as np

ox._version.__version__

'2.0.1'

In [2]:
def load_G_simplyfied():
    g = ox.load_graphml("tests/data/test_rng.ml")
    g = ox.simplify_graph(g)
    return g

G = load_G_simplyfied()

g_nodes = list(G.nodes())

Расчет с использованием чистого алгоритма

In [ ]:
from genesis.best_points import BestNodeHillClimbing
from genesis.core import MCLPBase, StateBase, MetricBase, BestPointsBase
from genesis.mclp import AdapterBLP2MCLP
from genesis.metrics import ArrivalTime
from genesis.states import FirstArrivalUnitState




#====================
# class AdapterBLP2MCLP(MCLPBase):

#     def __init__(self,
#                  blp_function:    BestPointsBase,
#                  state_function:  StateBase      = None,
#                  metric_function: MetricBase     = None,
#                  **kwargs):
#         self.blp_function = blp_function
#         super().__init__(state_function, metric_function, **kwargs)
   
#     def __call__(self,
#                  dynamic_nodes: dict,
#                  **kwargs) -> tuple[dict, int | float]:

#         # Проверка входящих данных
#         if dynamic_nodes is None:
#             raise ValueError("Аргумент `dynamic_nodes` должен быть словарем и не может быть равен None!")
#         if len(dynamic_nodes) < 1:
#             raise ValueError("Аргумент `dynamic_nodes` должен содержать не менее 1 узла!")

#         # Получение первого узла из словаря
#         start_point, start_key = list(dynamic_nodes.items())[0]

#         # Расчет лучшего узла с использованием BLP
#         best_node, best_metric = self.blp_function(start_point = start_point, **kwargs)

#         # Конвертация результата в результат метода MCLP
#         return {best_node: start_key}, best_metric
#====================







start_node = g_nodes[0]
print('Стартовый узел:', start_node)


# Расчет с использованием чистого алгоритма
bnhc = BestNodeHillClimbing(state_function = FirstArrivalUnitState(),
                            metric_function = ArrivalTime())

best_node_bnhc, best_metric_bnhc = bnhc(G, start_point = start_node)
print('Чистый BNHC:', best_node_bnhc, best_metric_bnhc)


# Расет с использованием адаптера
bnhc_mclp = AdapterBLP2MCLP(blp_function = bnhc)

best_node_bnhc_mclp, best_metric_bnhc_mclp_mclp = bnhc_mclp(env = G, dynamic_nodes = {start_node: 'test'})
print('BNHC(MCLP):', best_node_bnhc_mclp, best_metric_bnhc_mclp_mclp)


Стартовый узел: 290700365
Чистый BNHC: 426923808 7.789429840494647
BNHC(MCLP): {426923808: 'test'} 7.789429840494647


In [ ]:
from genesis.best_points import BestNodeBee

# start_node = g_nodes[300]
start_node = g_nodes[200]
print('Стартовый узел:', start_node)

# Расчет с использованием чистого алгоритма
bn = BestNodeBee(state_function = FirstArrivalUnitState(),
                   metric_function = ArrivalTime(),
                   )

best_node_bn, best_metric_bn = bn(G, start_point = start_node)
print('Чистый BNHC:', best_node_bn, best_metric_bn)


# Расет с использованием адаптера
bnmclp = AdapterBLP2MCLP(bn)

best_node_bnmclp, best_metric_bnmclp = bnmclp(env = G, dynamic_nodes = {start_node: 'test'})
print('BN ABC(MCLP):', best_node_bnmclp, best_metric_bnmclp)

Стартовый узел: 1555148537
Чистый BNHC: 426923837 7.6493893689184205
BN ABC(MCLP): {426923837: 'test'} 7.6493893689184205


In [ ]:
# Последовательность BLP(MCLP)


from genesis.best_points import BestNodeHillClimbing_maxMean, BestNodeMonkey



start_nodes = {
    g_nodes[0]: 'test',
}
print('Стартовый узел:', start_node)


m1 = AdapterBLP2MCLP(BestNodeHillClimbing(state_function=FirstArrivalUnitState(), metric_function=ArrivalTime()))
m2 = AdapterBLP2MCLP(BestNodeHillClimbing_maxMean(state_function=FirstArrivalUnitState()))
m3 = AdapterBLP2MCLP(
    BestNodeMonkey(state_function=FirstArrivalUnitState(), metric_function=ArrivalTime())
)
m4 = AdapterBLP2MCLP(BestNodeBee(state_function=FirstArrivalUnitState(),
                                  scouts_count=50,
                                  metric_function=ArrivalTime()))

best_nodes, metric_val = m1(env=G, dynamic_nodes = start_nodes)
print(best_nodes, metric_val)
best_nodes, metric_val = m2(env=G, dynamic_nodes = best_nodes)
print(best_nodes, metric_val)
best_nodes, metric_val = m3(env=G, dynamic_nodes = best_nodes)
print(best_nodes, metric_val)
best_nodes, metric_val = m4(env=G, dynamic_nodes = best_nodes)
print(best_nodes, metric_val)

Стартовый узел: 1555148537
{426923808: 'test'} 7.789429840494647
{426923808: 'test'} 7.789429840494647
{426923837: 'test'} 7.6493893689184205
{426923837: 'test'} 7.6493893689184205


In [ ]:
from genesis.best_points import BestNodesFull, BestNodesHalfDiameter
from genesis.utils import Progressbar


m0 = AdapterBLP2MCLP(
    BestNodesHalfDiameter()
)
m1 = AdapterBLP2MCLP(
    BestNodeHillClimbing(state_function=FirstArrivalUnitState(), metric_function=ArrivalTime())
    )
m2 = AdapterBLP2MCLP(
    BestNodeHillClimbing(state_function=FirstArrivalUnitState(), metric_function=ArrivalTime(f=np.max))
    )
m3 = AdapterBLP2MCLP(
    BestNodesFull(
        state_function=FirstArrivalUnitState(),
        metric_function=ArrivalTime(),
        node_calc_end_function=Progressbar(maxval=G.number_of_nodes())
        )
)

best_nodes, metric_val = m0(env=G, dynamic_nodes = start_nodes)
print(best_nodes, metric_val)
best_nodes, metric_val = m1(env=G, dynamic_nodes = best_nodes)
print(best_nodes, metric_val)
best_nodes, metric_val = m2(env=G, dynamic_nodes = best_nodes)
print(best_nodes, metric_val)
best_nodes, metric_val = m3(env=G, dynamic_nodes = best_nodes)
print(best_nodes, metric_val)

{4116035731: 'test'} None
{426923808: 'test'} 7.789429840494647
{4116049991: 'test'} 18.036191714285714
|##########| 100.0%
{426923837: 'test'} 7.6493893689184205


In [ ]:
m0 = AdapterBLP2MCLP(
    BestNodesHalfDiameter()
)
m1 = AdapterBLP2MCLP(
    BestNodeHillClimbing(state_function=FirstArrivalUnitState(), metric_function=ArrivalTime())
    )
m2 = AdapterBLP2MCLP(
    BestNodeHillClimbing(state_function=FirstArrivalUnitState(), metric_function=ArrivalTime(f=np.max))
    )
m3 = AdapterBLP2MCLP(
    BestNodesFull(
        state_function=FirstArrivalUnitState(),
        metric_function=ArrivalTime(),
        node_calc_end_function=Progressbar(maxval=G.number_of_nodes())
        )
)


best_nodes, metric_val = start_nodes, None
for m in [m0, m1, m2, m3]:
    best_nodes, metric_val = m(env=G, dynamic_nodes = start_nodes)
    print(best_nodes, metric_val)

best_nodes, metric_val

{4116035731: 'test'} None
{426923808: 'test'} 7.789429840494647
{2034401823: 'test'} 18.917574142857138
|##########| 100.0%
{426923837: 'test'} 7.6493893689184205


({426923837: 'test'}, np.float64(7.6493893689184205))

In [5]:
from genesis.best_points import BestNodeHillClimbing
from genesis.core import MCLPBase, StateBase, MetricBase, BestPointsBase
from genesis.mclp import AdapterBLP2MCLP
from genesis.metrics import ArrivalTime
from genesis.states import FirstArrivalUnitState


# G = create_G
start_node = list(G.nodes())[0]

bnhc0 = BestNodeHillClimbing(
    state_function=FirstArrivalUnitState(),
    metric_function=ArrivalTime())
best_node_bnhc0, best_metric_bnhc0 = bnhc0(G, start_point = start_node)

bnhc1  = AdapterBLP2MCLP(
    BestNodeHillClimbing(
        state_function=FirstArrivalUnitState(),
        metric_function=ArrivalTime()
        )
    )
best_node_bnhc1, best_metric_bnhc1 = bnhc1(G, dynamic_nodes={start_node: 'test'})